# Persistance and Fault tolerance implementation

In [29]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Literal
from pydantic import BaseModel, Field
from langchain_ollama import ChatOllama
from langgraph.checkpoint.memory import InMemorySaver

In [30]:
llm = ChatOllama(
    model='llama3.2:latest'
)

In [31]:
class JokeState(TypedDict):

    joke: str 
    topic: str 
    explanation: str 

In [32]:
checkpointer =  InMemorySaver()

In [33]:
def generate_joke(state: JokeState):

    prompt = f'generate a joke on the topic {state['topic']}'

    response = llm.invoke(prompt).content

    return {'joke':response}

In [34]:
import time
def interrupter(state: JokeState):

    print("step 3: interruption")
    time.sleep(20)
    return {'status':"interrupted"}

In [35]:
def generate_explanation(state: JokeState):

    prompt = f'write an explanation for the joke - {state['joke']}'

    response = llm.invoke(prompt).content

    return {'explanation':response}

In [36]:
graph = StateGraph(JokeState)


graph.add_node('generate_joke',generate_joke)
graph.add_node("interrupter",interrupter)
graph.add_node('generate_explanation',generate_explanation)

graph.add_edge(START,'generate_joke')
graph.add_edge('generate_joke','interrupter')
graph.add_edge('interrupter','generate_explanation')
graph.add_edge('generate_explanation',END)

joke_graph = graph.compile(checkpointer=checkpointer)

In [37]:
joke_graph

ValueError: Failed to reach https://mermaid.ink API while trying to render your graph after 1 retries. To resolve this issue:
1. Check your internet connection and try again
2. Try with higher retry settings: `draw_mermaid_png(..., max_retries=5, retry_delay=2.0)`
3. Use the Pyppeteer rendering method which will render your graph locally in a browser: `draw_mermaid_png(..., draw_method=MermaidDrawMethod.PYPPETEER)`

In [38]:
config1 = {"configurable":{"thread_id":"1"}}

try:
    joke_graph.invoke({'topic':'pizza'},config=config1)
except KeyboardInterrupt:
    print("keyboard interrupt occured...................")

step 3: interruption
keyboard interrupt occured...................


In [39]:
list(joke_graph.get_state_history(config1))


[StateSnapshot(values={'joke': 'Why was the pizza in a bad mood?\n\nBecause it was feeling crusty.', 'topic': 'pizza'}, next=('interrupter',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f13cb6d-f130-6117-8001-3935a72be88f'}}, metadata={'source': 'loop', 'step': 1, 'parents': {}}, created_at='2026-04-20T12:45:54.947487+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f13cb6d-e4f6-6e61-8000-0eae925a9bad'}}, tasks=(PregelTask(id='34892330-fbaa-305b-61fd-3a4b259f2360', name='interrupter', path=('__pregel_pull', 'interrupter'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'topic': 'pizza'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f13cb6d-e4f6-6e61-8000-0eae925a9bad'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2026-04-20T12:45:53.665782+00:00', parent_config={'con

In [40]:
joke_graph.invoke(None,config=config1)

step 3: interruption


{'joke': 'Why was the pizza in a bad mood?\n\nBecause it was feeling crusty.',
 'topic': 'pizza',
 'explanation': 'This joke is a play on words, using the multiple meanings of "crusty" to create humor.\n\nIn one sense, "crusty" can refer to the outer layer of bread that forms when dough is baked, which is also known as the crust of a pizza. In this context, the word "crusty" has a literal meaning related to the physical appearance and texture of the pizza.\n\nHowever, the phrase "feeling crusty" is an idiomatic expression that means feeling grumpy, irritable, or bitter. When applied to the pizza, it\'s a clever pun because the same word that describes the outer layer of the bread also describes the pizza\'s emotional state.\n\nThe joke relies on this unexpected twist on words, creating a humorous connection between the literal meaning of "crusty" (the physical characteristic of a pizza) and the figurative meaning (feeling grumpy). The punchline is unexpected and clever, making it amusi

In [ ]:
list(joke_graph.get_state_history(config1))


[StateSnapshot(values={'joke': 'Why was the pizza in a bad mood?\n\nBecause it was feeling crusty.', 'topic': 'pizza', 'explanation': 'This joke relies on a play on words, specifically a pun. The phrase "crusty" has a double meaning here:\n\n1. In a literal sense, crusty refers to the outer layer of a baked pizza, which is often crunchy and golden brown.\n2. However, the word "crusty" can also be used as an adjective to describe someone who is gruff, irritable, or short-tempered.\n\nThe joke exploits this dual meaning by using it to explain why the pizza is in a bad mood. The punchline, "feeling crusty," is a clever connection between the physical properties of a pizza and its emotional state. It\'s a lighthearted and humorous way to poke fun at the idea that even an inanimate object like a pizza can have a bad day due to its own characteristics.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f13cb56-0c7f-6ceb-8003-bed64d0b9b2a'}}, metada

In [ ]:
llm.invoke("what is pizza and how it is made?").content

'Pizza is a classic Italian dish that has become a staple in many cultures around the world. It\'s a type of flatbread typically topped with a variety of ingredients, such as cheese, meats, vegetables, and sauces.\n\n**The History of Pizza**\n\nThe origins of pizza date back to ancient Italy, specifically to the Neapolitan region in the 18th century. The word "pizza" was first mentioned in a Latin text from Gaeta, Italy, in 997 AD. However, it wasn\'t until the 19th century that modern pizza as we know it today emerged.\n\n**The Ingredients of Pizza**\n\nA traditional pizza consists of:\n\n1. **Crust**: A thin layer of dough made from wheat flour, water, yeast, salt, and sometimes olive oil.\n2. **Sauce**: A tomato-based sauce made from crushed tomatoes, garlic, olive oil, and herbs like basil and oregano.\n3. **Cheese**: Mozzarella is the most common type of cheese used on pizza, although other cheeses like Parmesan, Gorgonzola, or ricotta can also be used.\n4. **Toppings**: Various i